# Instalando bibliotecas e dependencias

In [1]:
!pip install pyspark

In [2]:
#from pyspark.sql.functions import input_file_name, split, col, lit, regexp_replace
#from datetime import datetime, timedelta
#from decimal import Decimal
#from pyspark.sql.functions import col, count, when, isnull, isnan, countDistinct, round, variance, stddev
#from pyspark.sql.types import NumericType, StringType
#from pyspark.sql import functions as F
import os
import time
from datetime import datetime

Inicializando o Spark

In [3]:
from pyspark.sql import SparkSession
# Inicializando a sessão
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Camada Silver") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

# Verificando se funcionou
print("Sessão Spark criada com sucesso!")
spark

Sessão Spark criada com sucesso!


Instanciando o Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Criar uma função log para registrar a data de importação dos dados

In [5]:
#Criar uma função log para registrar a data de importação dos dados
def log():
    return datetime.now().strftime('%d/%m/%Y-%H:%M:%S')
dt_proc = log()
#current_date = datetime.now().strftime('%d/%m/%Y')

dt_proc

'06/01/2026-19:13:13'

# Leitura de um arquivo para testar a conexão

In [6]:
file_path = '/content/drive/MyDrive/hackathon_pod_2025/database/raw/book_pagamento/dados_pagamento/part-00000-1541d958-609c-45e7-aa12-861b49d5d79d-c000.snappy.parquet'

df_pagamento = spark.read.parquet (file_path, header=True, inferSchema=True)
df_pagamento.createOrReplaceTempView("df_pagamento_um_parquet")

df_pagamento.show()
#spark.sql("""
#SELECT distinct dw_area FROM df_pagamento_um_parquet
#""").show()


+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+--------------------

#Leitura de todos os arquivo parquet e registro da view df_pagamentos para o spark SQL

In [7]:
target_directory = '/content/drive/MyDrive/hackathon_pod_2025/database/raw/book_pagamento/dados_pagamento'

parquet_files = [os.path.join(target_directory,f) for f in os.listdir(target_directory) if f.endswith('.parquet')]

if not parquet_files:
    print(f"No .parquet files found in the directory: {target_directory}")
else:
  print(f"Loading {len(parquet_files)} parquet files from: {target_directory}")
  df_pagamento= spark.read.parquet(*parquet_files, header=True, inferSchema=True)
  df_pagamento.createOrReplaceTempView("df_pagamento")
  df_pagamento.show()

Loading 10 parquet files from: /content/drive/MyDrive/hackathon_pod_2025/database/raw/book_pagamento/dados_pagamento
+-----------+------------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+-------------

In [8]:
#Quantidade de linhas
df_pagamento.count()

21829628

In [9]:
#Estrutura dos dados
df_pagamento.printSchema()

root
 |-- NUM_CPF: string (nullable = true)
 |-- DAT_STATUS_FATURA: string (nullable = true)
 |-- CONTRATO: string (nullable = true)
 |-- SEQ_FATURA: string (nullable = true)
 |-- NUM_SUB_SEQ_FATURA: string (nullable = true)
 |-- NUM_CREDITO_SEQ: string (nullable = true)
 |-- DW_TIPO_FATURA: string (nullable = true)
 |-- IND_STATUS_FATURA: string (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- DW_AREA: string (nullable = true)
 |-- DW_UN_NEGOCIO: string (nullable = true)
 |-- DW_FORMA_PAGAMENTO: string (nullable = true)
 |-- VAL_PAGAMENTO_FATURA: string (nullable = true)
 |-- DAT_CRIACAO_DW: string (nullable = true)
 |-- DW_BANCO: string (nullable = true)
 |-- DW_TIPO_PAGAMENTO: string (nullable = true)
 |-- NUM_BANCO_PAGAMENTO: string (nullable = true)
 |-- NUM_AGENCIA_PAGAMENTO: string (nullable = true)
 |-- NUM_CC_PAGAMENTO: string (nullable = true)
 |-- DW_MOTIVO_ESTORNO: string (nullable = true)
 |-- VAL_DESCONTO_ITEM: string (nullable = true)
 |-- VAL_PAGAMEN

In [14]:
#testando os campos para identificar se temos diferença nas horas
spark.sql("""
SELECT
--DAT_CRIACAO_DW,
--TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy:HH:mm:ss') AS DAT_CRIACAO_DW_Ajustado
distinct
--substr(DAT_ATIVIDADE_CREDITO, -8, 10) as ativ_credito
--substr(DAT_VENCIMENTO_CREDITO, -8, 10) as vencimento
--substr(DAT_STATUS_PAGAMENTO, -8, 10) as DAT_STATUS_PAGAMENTO
--substr(DAT_DEPOSITO_ATIVIDADE, -8, 10) as DAT_DEPOSITO_ATIVIDADE
--substr(DAT_BAIXA_ATIVIDADE, -8, 10) as DAT_BAIXA_ATIVIDADE
--substr(DAT_STATUS_FATURA, -8, 10) as DAT_STATUS_FATURA
DW_BANCO
FROM df_pagamento
""").show(10)



+--------+
|DW_BANCO|
+--------+
|    1669|
|    1361|
|    1349|
|    1377|
|    1642|
|    1655|
|    1977|
|    1982|
|    1696|
|    1965|
+--------+
only showing top 10 rows


In [11]:
dt_proc = log()

df_pagamento_ajustado = spark.sql(f"""
SELECT
'{dt_proc}' as DATA_IMPORT,
CAST(NUM_CPF AS STRING) AS NUM_CPF,
TO_CHAR(TO_DATE(DAT_STATUS_FATURA, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy') AS DAT_STATUS_FATURA,
CAST(CONTRATO AS STRING) AS CONTRATO,
CAST(SEQ_FATURA AS INT) AS SEQ_FATURA,
CAST(NUM_SUB_SEQ_FATURA AS INT) AS NUM_SUB_SEQ_FATURA,
CAST(NUM_CREDITO_SEQ AS INT) AS NUM_CREDITO_SEQ,
CAST(DW_TIPO_FATURA AS INT) AS DW_TIPO_FATURA,
CAST(IND_STATUS_FATURA AS STRING) AS IND_STATUS_FATURA,
CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
CAST(DW_AREA AS INT) AS DW_AREA,
CAST(DW_UN_NEGOCIO AS INT) AS DW_UN_NEGOCIO,
CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
CAST(VAL_PAGAMENTO_FATURA AS FLOAT) AS VAL_PAGAMENTO_FATURA,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_DW, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy:HH:mm:ss') AS DAT_CRIACAO_DW,
CAST(DW_BANCO AS STRING) AS DW_BANCO,
CAST(DW_TIPO_PAGAMENTO AS STRING) AS DW_TIPO_PAGAMENTO,
CAST(NUM_BANCO_PAGAMENTO AS STRING) AS NUM_BANCO_PAGAMENTO,
CAST(NUM_AGENCIA_PAGAMENTO AS STRING) AS NUM_AGENCIA_PAGAMENTO,
CAST(NUM_CC_PAGAMENTO AS STRING) AS NUM_CC_PAGAMENTO,
CAST(DW_MOTIVO_ESTORNO AS STRING) AS DW_MOTIVO_ESTORNO,
CAST(VAL_DESCONTO_ITEM AS FLOAT) AS VAL_DESCONTO_ITEM,
CAST(VAL_PAGAMENTO_ITEM AS FLOAT) AS VAL_PAGAMENTO_ITEM,
CAST(VAL_JUROS_MULTAS_ITEM AS FLOAT) AS VAL_JUROS_MULTAS_ITEM,
CAST(VAL_MULTA_EQUIP_ITEM AS FLOAT) AS VAL_MULTA_EQUIP_ITEM,
CAST(VAL_MULTA_EQUIP_TOTAL AS FLOAT) AS VAL_MULTA_EQUIP_TOTAL,
CAST(VAL_MULTA_FID_ITEM AS FLOAT) AS VAL_MULTA_FID_ITEM,
CAST(COD_ORIGEM_NETUNO AS STRING) AS COD_ORIGEM_NETUNO,
CAST(COD_CONTA_ATIVIDADE AS STRING) AS COD_CONTA_ATIVIDADE,
CAST(SEQ_ENTIDADE_ATIVIDADE AS INT) AS SEQ_ENTIDADE_ATIVIDADE,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy:HH:mm:ss') AS DAT_CRIACAO_ATIVIDADE,
TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/:HH:mm:ss') AS DAT_ATUALIZACAO_ATIVIDADE,
CAST(COD_LOGIN_OPERADOR_ATIVIDADE AS STRING) AS COD_LOGIN_OPERADOR_ATIVIDADE,
CAST(COD_ATIVIDADE AS STRING) AS COD_ATIVIDADE,
CAST(COD_RAZAO_ATIVIDADE AS STRING) AS COD_RAZAO_ATIVIDADE,
TO_CHAR(TO_DATE(DAT_BAIXA_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy') AS DAT_BAIXA_ATIVIDADE,
CAST(VAL_BAIXA_ATIVIDADE AS FLOAT) AS VAL_BAIXA_ATIVIDADE,
TO_CHAR(TO_DATE(DAT_DEPOSITO_ATIVIDADE, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy') AS DAT_DEPOSITO_ATIVIDADE,
CAST(COD_FUNDO_ATIVIDADE AS STRING) AS COD_FUNDO_ATIVIDADE,
CAST(COD_BANCO_ATIVIDADE AS STRING) AS COD_BANCO_ATIVIDADE,
CAST(NUM_CONTA_ATIVIDADE AS STRING) AS NUM_CONTA_ATIVIDADE,
CAST(COD_AGENCIA_ATIVIDADE AS STRING) AS COD_AGENCIA_ATIVIDADE,
CAST(SEQ_ENTIDADE_PAGAMENTO AS INT) AS SEQ_ENTIDADE_PAGAMENTO,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy:HH:mm:ss') AS DAT_CRIACAO_PAGAMENTO,
TO_CHAR(TO_TIMESTAMP(DAT_ATUALIZACAO_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy:HH:mm:ss') AS DAT_ATUALIZACAO_PAGAMENTO,
CAST(COD_LOGIN_PAGAMENTO AS STRING) AS COD_LOGIN_PAGAMENTO,
CAST(COD_FORMA_PAGAMENTO AS STRING) AS COD_FORMA_PAGAMENTO,
CAST(VAL_ORIGINAL_PAGAMENTO AS FLOAT) AS VAL_ORIGINAL_PAGAMENTO,
CAST(NUM_FATURA_PAGAMENTO AS STRING) AS NUM_FATURA_PAGAMENTO,
CAST(COD_TIPO_PAGAMENTO AS STRING) AS COD_TIPO_PAGAMENTO,
CAST(DSC_NOME_BANCO_PAGAMENTO AS STRING) AS DSC_NOME_BANCO_PAGAMENTO,
CAST(SEQ_ARQUIVO_PAGAMENTO AS INT) AS SEQ_ARQUIVO_PAGAMENTO,
CAST(NUM_PARCELA_PAGAMENTO AS INT) AS NUM_PARCELA_PAGAMENTO,
CAST(NUM_AGRUPADOR_PAGAMENTO AS INT) AS NUM_AGRUPADOR_PAGAMENTO,
CAST(DSC_PAGAMENTO AS STRING) AS DSC_PAGAMENTO,
CAST(VAL_ATUAL_PAGAMENTO AS FLOAT) AS VAL_ATUAL_PAGAMENTO,
CAST(COD_METODO_PAGAMENTO AS INT) AS COD_METODO_PAGAMENTO,
CAST(IND_STATUS_PAGAMENTO AS STRING) AS IND_STATUS_PAGAMENTO,
TO_CHAR(TO_DATE(DAT_STATUS_PAGAMENTO, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy') AS DAT_STATUS_PAGAMENTO,
CAST(COD_ARQUIVO_PAGAMENTO AS STRING) AS COD_ARQUIVO_PAGAMENTO,
CAST(COD_NETUNO_PAGAMENTO AS STRING) AS COD_NETUNO_PAGAMENTO,
TO_CHAR(TO_TIMESTAMP(DAT_CRIACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy:HH:mm:ss') AS DAT_CRIACAO_CREDITO,
TO_CHAR(TO_DATE(DAT_ATUALIZACAO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy') AS DAT_ATUALIZACAO_CREDITO,
CAST(COD_LOGIN_CREDITO AS STRING) AS COD_LOGIN_CREDITO,
CAST(VAL_PAGAMENTO_CREDITO AS FLOAT) AS VAL_PAGAMENTO_CREDITO,
CAST(IND_TIPO_CREDITO AS STRING) AS IND_TIPO_CREDITO,
CAST(SEQ_PAGAMENTO_CREDITO AS INT) AS SEQ_PAGAMENTO_CREDITO,
CAST(SEQ_FATURA_CREDITO AS INT) AS SEQ_FATURA_CREDITO,
CAST(COD_ALOCACAO_CREDITO AS STRING) AS COD_ALOCACAO_CREDITO,
CAST(COD_DESALOCACAO_CREDITO AS STRING) AS COD_DESALOCACAO_CREDITO,
CAST(SEQ_ENTIDADE_CREDITO AS INT) AS SEQ_ENTIDADE_CREDITO,
CAST(COD_TIPO_FATURA AS STRING) AS COD_TIPO_FATURA,
TO_CHAR(TO_DATE(DAT_ATIVIDADE_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy') AS DAT_ATIVIDADE_CREDITO,
TO_CHAR(TO_DATE(DAT_VENCIMENTO_CREDITO, 'ddMMMyyyy:HH:mm:ss'),'dd/MM/yyyy') AS DAT_VENCIMENTO_CREDITO
FROM df_pagamento
""")
df_pagamento_ajustado.createOrReplaceTempView("df_pagamento_ajustado")
df_pagamento_ajustado.show()

+-------------------+-----------+-----------------+---------+----------+------------------+---------------+--------------+-----------------+--------------+-------+-------------+------------------+--------------------+-------------------+--------+-----------------+-------------------+---------------------+----------------+-----------------+-----------------+------------------+---------------------+--------------------+---------------------+------------------+-----------------+-------------------+----------------------+---------------------+-------------------------+----------------------------+-------------+-------------------+-------------------+-------------------+----------------------+-------------------+-------------------+-------------------+---------------------+----------------------+---------------------+-------------------------+-------------------+-------------------+----------------------+--------------------+------------------+------------------------+---------------------+

In [12]:
df_pagamento_ajustado.printSchema()

root
 |-- DATA_IMPORT: string (nullable = false)
 |-- NUM_CPF: string (nullable = true)
 |-- DAT_STATUS_FATURA: string (nullable = true)
 |-- CONTRATO: string (nullable = true)
 |-- SEQ_FATURA: integer (nullable = true)
 |-- NUM_SUB_SEQ_FATURA: integer (nullable = true)
 |-- NUM_CREDITO_SEQ: integer (nullable = true)
 |-- DW_TIPO_FATURA: integer (nullable = true)
 |-- IND_STATUS_FATURA: string (nullable = true)
 |-- DW_NUM_CLIENTE: string (nullable = true)
 |-- DW_AREA: integer (nullable = true)
 |-- DW_UN_NEGOCIO: integer (nullable = true)
 |-- DW_FORMA_PAGAMENTO: integer (nullable = true)
 |-- VAL_PAGAMENTO_FATURA: float (nullable = true)
 |-- DAT_CRIACAO_DW: string (nullable = true)
 |-- DW_BANCO: string (nullable = true)
 |-- DW_TIPO_PAGAMENTO: string (nullable = true)
 |-- NUM_BANCO_PAGAMENTO: string (nullable = true)
 |-- NUM_AGENCIA_PAGAMENTO: string (nullable = true)
 |-- NUM_CC_PAGAMENTO: string (nullable = true)
 |-- DW_MOTIVO_ESTORNO: string (nullable = true)
 |-- VAL_DESCON

#Salvar o arquivo na camada Silver

In [ ]:
#Avaliar como faremos a partição
  #.partitionBy("DATA_IMPORT") \

target_directory_write = "/content/drive/MyDrive/hackathon_pod_2025/database/silver/book_pagamento/dados_pagamento"

df_pagamento_ajustado.write \
  .mode("append") \
  .parquet(target_directory_write)

In [ ]:
parquet_files = [os.path.join(target_directory_write,f) for f in os.listdir(target_directory_write) if f.endswith('.parquet')]

if not parquet_files:
    print(f"No .parquet files found in the directory: {target_directory_write}")
else:
  print(f"Loading {len(parquet_files)} parquet files from: {target_directory_write}")
  df_silver= spark.read.parquet(*parquet_files, header=True, inferSchema=True)
  df_silver.createOrReplaceTempView("df_silver")
  df_silver.show()